In [1]:
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt

In [2]:
# загружаем данные

file_path = 'sister.csv'

data = pd.read_csv(file_path, header=None, encoding='utf-8')
data.columns = data.iloc[0]
data = data.drop(0).reset_index(drop=True)

data.head()


,Конфигурация,Локализация,Ориентация (ладонь),Направление пальцев,Траекторное + / Локальное -,Направление движения,Характер движения,Повтор
0,1,подбородок - нейтральная,ребром,вперед,траекторный,вниз,по прямой,да
1,П,подбородок - нейтральная,ребром,вперед,траекторный,вниз,по прямой,нет
2,П,подбородок - нейтральная,ребром,вперед,траекторный,вниз,по прямой,да
3,1,подбородок - нейтральная,ребром,вперед,траекторный,вниз,по прямой,нет
4,1,подбородок - нейтральная,ребром,вперед,траекторный,вниз,по прямой,нет


In [3]:
# создаем граф, в котором каждый вариант жеста - вершина

G = nx.Graph()
for i in range(len(data)):
    G.add_node(i+1)

In [4]:
# удаляем параметры, который не рассматриваем при построении графа

data = data.drop('Ориентация (ладонь)', axis=1)
data = data.drop('Траекторное + / Локальное -', axis=1)
data = data.drop('Направление пальцев', axis=1)

In [5]:
data

,Конфигурация,Локализация,Направление движения,Характер движения,Повтор
0,1,подбородок - нейтральная,вниз,по прямой,да
1,П,подбородок - нейтральная,вниз,по прямой,нет
2,П,подбородок - нейтральная,вниз,по прямой,да
3,1,подбородок - нейтральная,вниз,по прямой,нет
4,1,подбородок - нейтральная,вниз,по прямой,нет
5,П,плечо,вниз,по кругу,нет
6,Э,нейтральная,вниз,по прямой,нет


In [6]:
# функция для подсчета различий между двумя вариантами жеста

def count_differences(row1, row2):
    count = 0
    for i in range(len(row1)):
        if row1[i] != row2[i]:
            count += 1
    return count

In [7]:
# если различие ровно 1 или его нет, то проводим ребро

for i in range(len(data)):
    for j in range(i+1, len(data)):
        if count_differences(data.iloc[i], data.iloc[j]) == 1 or count_differences(data.iloc[i], data.iloc[j]) == 0:
            G.add_edge(i+1, j+1)

/tmp/ipykernel_10941/697979470.py:6: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  if row1[i] != row2[i]:


In [8]:
# выделяем из графа все циклы

G_directed = nx.DiGraph(G)
c = nx.simple_cycles(G_directed)
cycle_sets = []
for i in c:
    if set(i) not in cycle_sets:
        cycle_sets.append(set(i))

cycle_unique = []
for i in cycle_sets:
    if len(i) > 2:
        ind = 0
        for j in cycle_sets:
            if i.issubset(j) and i!=j:
                ind = 1
        if ind == 0 :
            cycle_unique.append(i)

In [9]:
# находим все вершины, которые участвуют в циклах

cycle_unique_nodes = set()
for i in cycle_unique:
    for j in i:
        cycle_unique_nodes.add(j)

cycle_unique_nodes

{1, 2, 3, 4, 5}

In [10]:
# находим все прочие вершины
other_nodes = []
for n in G.nodes:
    if n not in cycle_unique_nodes:
        other_nodes.append(n)

In [11]:
# список всех лексем
all_lexemas = []

# добавляем все циклы в лексемы
for l in cycle_unique:
    all_lexemas.append(l)

# находим все прочие вершины, у которых есть ребра. Те у которых нет добавляем как отдельные лексемы
other_lexemas_nodes = []
for n in other_nodes:
    if len(G[n]) == 0:
        all_lexemas.append(set([n]))
    else:
        other_lexemas_nodes.append(n)

for e in G.edges:
    if e[0] in other_lexemas_nodes or e[1] in other_lexemas_nodes:
        all_lexemas.append(set(e))
all_lexemas

[{1, 2, 3, 4, 5}, {6}, {7}]

In [13]:
edge_view = G.edges

In [14]:
edges = list(edge_view)

# Формирование строки в формате R
r_code = "graph(c(\n  " + ",\n  ".join([f"{u},{v}" for u, v in edges]) + "\n))"

print(r_code)

graph(c(
  1,3,
  1,4,
  1,5,
  2,3,
  2,4,
  2,5,
  4,5
))


In [15]:
data = all_lexemas
result = "list(\n  " + ",\n  ".join([f"c({', '.join(map(str, sorted(s)))})" for s in data]) + "\n)"

print(result)

list(
  c(1, 2, 3, 4, 5),
  c(6),
  c(7)
)
